# AgroVision – Climate-Smart Agricultural Advisory System
## Gweru, Midlands, Zimbabwe


- Real NASA POWER weather data for Gweru
- Knowledge-base + PyTorch MLP for crop suitability
- Planting window from rainfall onset detection
- Pest risk rules based on temp + humidity + crop stage
- Reproducible, exportable model, ready for FastAPI / Streamlit deployment

Run this notebook on Google Colab (GPU optional – model is small).

## 1. Setup

In [ ]:
!pip install -q torch pandas scikit-learn matplotlib seaborn plotly requests

import os
import sys
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from pathlib import Path
from datetime import datetime

print('Torch', torch.__version__)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

## 2. Fetch real NASA POWER data for Gweru
Coordinates: 19.45°S, 29.82°E

In [ ]:
lat, lon = -19.45, 29.82
start, end = '20150101', '20241231'
params = 'T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,RH2M,WS10M'
url = f'https://power.larc.nasa.gov/api/temporal/daily/point?parameters={params}&community=AG&longitude={lon}&latitude={lat}&start={start}&end={end}&format=JSON'

print('Fetching NASA POWER...')
r = requests.get(url, timeout=120)
data = r.json()
param_data = data['properties']['parameter']
df = pd.DataFrame(param_data)
df.index = pd.to_datetime(df.index, format='%Y%m%d')
df = df.reset_index().rename(columns={'index': 'date'}).sort_values('date').reset_index(drop=True)

for col in ['T2M','T2M_MAX','T2M_MIN','PRECTOTCORR','RH2M','WS10M']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[df[col] < -900, col] = np.nan

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['doy'] = df['date'].dt.dayofyear

print(df.shape)
df.head()

## 3. Exploratory Data Analysis

In [ ]:
yearly = df.groupby('year').agg(
    total_rain_mm=('PRECTOTCORR', 'sum'),
    mean_temp=('T2M', 'mean'),
    rainy_days=('PRECTOTCORR', lambda x: (x >= 1).sum())
).reset_index()

print('Long-term average annual rainfall (mm):', yearly['total_rain_mm'].mean().round(1))
print('Mean temperature (°C):', df['T2M'].mean().round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(yearly['year'], yearly['total_rain_mm'], color='steelblue')
axes[0].axhline(yearly['total_rain_mm'].mean(), color='red', ls='--', label='Mean')
axes[0].set_title('Annual Rainfall – Gweru (NASA POWER)')
axes[0].set_ylabel('mm')
axes[0].legend()

clim = df.groupby('month')['PRECTOTCORR'].mean() * 30.4
axes[1].bar(range(1,13), clim.values, color='teal')
axes[1].set_xticks(range(1,13))
axes[1].set_title('Approx. Monthly Rainfall Climatology')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('mm')
plt.tight_layout()
plt.show()

## 4. Crop Knowledge Base + Feature Engineering
(Same logic as production `crop_knowledge.py`)

In [ ]:
# Simplified crop dict for notebook self-containment
CROPS = {
    'maize': {'drought_tolerance': 0.4, 'input_requirement': 'high'},
    'sorghum': {'drought_tolerance': 0.85, 'input_requirement': 'low'},
    'pearl_millet': {'drought_tolerance': 0.95, 'input_requirement': 'low'},
    'finger_millet': {'drought_tolerance': 0.75, 'input_requirement': 'low'},
    'groundnut': {'drought_tolerance': 0.55, 'input_requirement': 'medium'},
    'cowpea': {'drought_tolerance': 0.80, 'input_requirement': 'low'},
    'sugar_bean': {'drought_tolerance': 0.45, 'input_requirement': 'medium'},
    'sunflower': {'drought_tolerance': 0.70, 'input_requirement': 'medium'},
}
CROP_TO_IDX = {k: i for i, k in enumerate(CROPS)}
print('Crops:', list(CROPS.keys()))

## 5. PyTorch Crop Suitability Network

In [ ]:
class CropSuitabilityNet(nn.Module):
    def __init__(self, n_crops=8, emb_dim=8, hidden=[64, 32], dropout=0.15):
        super().__init__()
        self.crop_emb = nn.Embedding(n_crops, emb_dim)
        layers = []
        prev = 10 + emb_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(inplace=True), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x, crop_idx):
        emb = self.crop_emb(crop_idx)
        return self.mlp(torch.cat([x, emb], dim=-1)).squeeze(-1)

model = CropSuitabilityNet().to(device)
print(model)
print('Parameters:', sum(p.numel() for p in model.parameters()))

## 6. Synthetic training data (knowledge-base soft labels + noise)
Standard approach when large labelled farm outcome datasets are unavailable.

In [ ]:
def build_feature_vector(soil_pH, annual_rain, mean_temp, risk_pref, input_access, drought_tol, input_req_high):
    pH_n = (soil_pH - 4.0) / 3.5
    rain_n = (annual_rain - 300) / 700
    temp_n = (mean_temp - 10) / 25
    return [
        pH_n, rain_n, temp_n,
        1.0 if risk_pref == 'risk_averse' else 0.0,
        1.0 if risk_pref == 'risk_seeking' else 0.0,
        1.0 if input_access == 'low' else 0.0,
        1.0 if input_access == 'medium' else 0.0,
        1.0 if input_access == 'high' else 0.0,
        drought_tol, input_req_high,
    ]

# Simple composite score for soft labels (mirrors production knowledge base)
def soft_label(crop_key, soil_pH, rain, temp, risk, inp):
    # Very simplified scoring for notebook
    dt = CROPS[crop_key]['drought_tolerance']
    score = 0.5 + 0.3 * dt
    if risk == 'risk_averse':
        score += 0.2 * dt
    if inp == 'low' and CROPS[crop_key]['input_requirement'] == 'high':
        score -= 0.25
    if 5.0 <= soil_pH <= 6.0:
        score += 0.1
    if 500 <= rain <= 800:
        score += 0.1
    return float(np.clip(score + np.random.normal(0, 0.05), 0.01, 0.99))

rng = np.random.default_rng(42)
X_list, c_list, y_list = [], [], []
for _ in range(4000):
    soil_pH = rng.uniform(4.2, 7.0)
    rain = rng.uniform(350, 950)
    temp = rng.uniform(15, 24)
    risk = rng.choice(['risk_averse', 'neutral', 'risk_seeking'])
    inp = rng.choice(['low', 'medium', 'high'])
    for key in CROPS:
        feats = build_feature_vector(soil_pH, rain, temp, risk, inp,
                                    CROPS[key]['drought_tolerance'],
                                    1.0 if CROPS[key]['input_requirement'] == 'high' else 0.0)
        X_list.append(feats)
        c_list.append(CROP_TO_IDX[key])
        y_list.append(soft_label(key, soil_pH, rain, temp, risk, inp))

X = torch.tensor(X_list, dtype=torch.float32)
crop_idx = torch.tensor(c_list, dtype=torch.long)
y = torch.tensor(y_list, dtype=torch.float32)
print(X.shape, y.shape)

## 7. Train

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import copy

n = len(X)
perm = torch.randperm(n)
train_idx, val_idx = perm[:int(0.8*n)], perm[int(0.8*n):]
train_loader = DataLoader(TensorDataset(X[train_idx], crop_idx[train_idx], y[train_idx]), batch_size=256, shuffle=True)
val_loader = DataLoader(TensorDataset(X[val_idx], crop_idx[val_idx], y[val_idx]), batch_size=256)

opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)
crit = nn.MSELoss()

best_val, best_state = float('inf'), None
history = {'train': [], 'val': []}

for epoch in range(1, 31):
    model.train()
    tr_loss = 0
    for xb, cb, yb in train_loader:
        xb, cb, yb = xb.to(device), cb.to(device), yb.to(device)
        opt.zero_grad()
        pred = torch.sigmoid(model(xb, cb))
        loss = crit(pred, yb)
        loss.backward()
        opt.step()
        tr_loss += loss.item() * len(xb)
    tr_loss /= len(train_idx)

    model.eval()
    va_loss = 0
    with torch.no_grad():
        for xb, cb, yb in val_loader:
            xb, cb, yb = xb.to(device), cb.to(device), yb.to(device)
            pred = torch.sigmoid(model(xb, cb))
            va_loss += crit(pred, yb).item() * len(xb)
    va_loss /= len(val_idx)
    sched.step()
    history['train'].append(tr_loss)
    history['val'].append(va_loss)
    if va_loss < best_val:
        best_val = va_loss
        best_state = copy.deepcopy(model.state_dict())
    if epoch % 5 == 0:
        print(f'Epoch {epoch:2d}  train {tr_loss:.4f}  val {va_loss:.4f}')

model.load_state_dict(best_state)
print('Best val MSE:', round(best_val, 4))

plt.plot(history['train'], label='train')
plt.plot(history['val'], label='val')
plt.legend(); plt.title('Training curve'); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.show()

## 8. Export model (TorchScript ready for serving)

In [ ]:
model.eval()
torch.save(model.state_dict(), 'crop_suitability_net.pt')
print('Saved state_dict → crop_suitability_net.pt')

# Optional TorchScript
example_x = torch.randn(1, 10)
example_c = torch.tensor([0])
traced = torch.jit.trace(model.cpu(), (example_x, example_c))
traced.save('crop_suitability_net_ts.pt')
print('Saved TorchScript → crop_suitability_net_ts.pt')

## 9. Quick inference demo

In [ ]:
def predict_all(model, soil_pH=5.2, rain=650, temp=19.5, risk='risk_averse', inp='low'):
    model.eval()
    scores = {}
    with torch.no_grad():
        for key, idx in CROP_TO_IDX.items():
            feats = build_feature_vector(soil_pH, rain, temp, risk, inp,
                                        CROPS[key]['drought_tolerance'],
                                        1.0 if CROPS[key]['input_requirement'] == 'high' else 0.0)
            x = torch.tensor([feats], dtype=torch.float32)
            c = torch.tensor([idx])
            scores[key] = torch.sigmoid(model(x, c)).item()
    return dict(sorted(scores.items(), key=lambda kv: -kv[1]))

print('Risk-averse + low inputs:')
print(predict_all(model))
print()
print('Risk-seeking + high inputs:')
print(predict_all(model, risk='risk_seeking', inp='high'))

## Next steps for production
1. Copy `crop_suitability_net.pt` into the `models/` folder of the repo.
2. Run the Streamlit app: `streamlit run app/streamlit_app.py`
3. Or expose via FastAPI for mobile clients.
4. Continuously improve by collecting real farmer feedback / yields as labels.